In [3]:
import langchain, langchain_core
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
load_dotenv()

True

In [4]:
llm = ChatOpenAI(model='gpt-4o-mini')

In [10]:
llm.invoke('Hi').content

'Hello! How can I assist you today?'

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [32]:
# prompt = 
prompt = ChatPromptTemplate.from_template(
    'Reply in one sentence: {question}'
)

parser = StrOutputParser()
chain = prompt | llm | parser

# https://python.langchain.com/docs/concepts/lcel/

result = chain.invoke('What is LLM')

In [22]:
classify_prompt = ChatPromptTemplate.from_template(
    'Classify this email in ONE word (Billing/Technical/Spam/Other):\n'
    'Subject: {subject}\nBody: {body}'
)

# chain
clasify_chain = classify_prompt | llm | parser

In [23]:
emails = [
    {'subject': 'Payment failed',   'body': 'Charged twice this month.'},
    {'subject': 'App crash',        'body': 'Crashes on Settings page.'},
    {'subject': 'Add Slack?',       'body': 'Want Slack notifications.'},
    {'subject': 'WIN FREE IPHONE',  'body': 'Click here NOW!!!'},
    {'subject': 'Invoice question', 'body': 'Can I get annual invoice?'},
]

In [25]:
emails[0]

{'subject': 'Payment failed', 'body': 'Charged twice this month.'}

In [27]:
# invoke / batch /stream
# compare for loop vs batch : time ??

import time
start = time.time()
loop_result = []
for email in emails:
    loop_result.append(clasify_chain.invoke(email))

loop_time = time.time()-start
print(loop_time)

4.657979965209961


In [28]:
loop_result

['Billing', 'Technical', 'Other', 'Spam', 'Billing']

In [29]:
batch_results = clasify_chain.batch(emails,
                                    config={'max_concurrency':3})

In [30]:
batch_results

['Billing', 'Technical', 'Other', 'Spam', 'Billing']

In [33]:
clasify_chain.with_retry(
    stop_after_attempt=3,
    wait_exponential_jitter=True
)

RunnableRetry(bound=ChatPromptTemplate(input_variables=['body', 'subject'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['body', 'subject'], input_types={}, partial_variables={}, template='Classify this email in ONE word (Billing/Technical/Spam/Other):\nSubject: {subject}\nBody: {body}'), additional_kwargs={})])
| ChatOpenAI(output_version=None, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice

In [34]:
clasify_chain.batch(emails)

['Billing', 'Technical', 'Other', 'Spam', 'Billing']